# Vision Track — Phase 3

**Goal:** Take a raw guitar video → RT-DETR fretboard detection → MediaPipe hand tracking → (string, fret) coordinates per frame.

**Prerequisites:**
- `pip install mediapipe opencv-python transformers torch torchvision`
- A sample video in `../video_recording/` (MP4 or MOV)
- For RT-DETR fine-tuning: a Roboflow dataset with fretboard bounding box annotations

Stages:
1. Extract and visualize video frames
2. Run MediaPipe hand tracking on a sample frame
3. Run RT-DETR fretboard detection (pre-trained, then fine-tuned)
4. Map fingertip coordinates to string/fret positions
5. Run the full video and produce per-frame vision results

## 1. Load video and extract frames

**What you're looking for in a sample frame:**
- Fretboard clearly visible and mostly horizontal
- Fingers visible without major motion blur
- Good lighting — MediaPipe struggles with dark or overexposed frames

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

VIDEO_PATH = '../video_recording/test_video.mp4'  # ← change this

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f'FPS:          {fps:.1f}')
print(f'Total frames: {total_frames}')
print(f'Duration:     {total_frames / fps:.1f}s')
print(f'Resolution:   {width}x{height}')

# Extract one frame per second for inspection
sample_frames = []
for i in range(min(6, int(total_frames / fps))):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(i * fps))
    ret, frame = cap.read()
    if ret:
        sample_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
cap.release()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, (ax, frame) in enumerate(zip(axes.flat, sample_frames)):
    ax.imshow(frame)
    ax.set_title(f't={i}s')
    ax.axis('off')
plt.suptitle('Sample frames (1 per second)', fontsize=14)
plt.tight_layout()
plt.show()

## 2. MediaPipe hand tracking

**What MediaPipe Hands does:** Given an image, it detects up to N hands and returns 21 3D landmark points per hand. The landmarks are normalized (x, y in [0,1], z relative to wrist depth).

**Landmark indices you care about most:**
```
4  = thumb tip        8  = index tip
12 = middle tip       16 = ring tip        20 = pinky tip
5  = index base       9  = middle base     13 = ring base      17 = pinky base
0  = wrist
```
The tips tell you which fret is being pressed. The bases (knuckles) tell you where the hand position is centered.

**Handedness note:** MediaPipe's "Left"/"Right" labels are from the *camera's* perspective, which is the *mirror* of the player's perspective. A right-handed player's fretting hand will appear as "Right" in the image but MediaPipe may label it "Left". Verify this on your own footage.

In [ ]:
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_draw  = mp.solutions.drawing_utils

# Use the first sample frame for testing
test_frame_rgb = sample_frames[0]  # RGB uint8
test_frame_bgr = cv2.cvtColor(test_frame_rgb, cv2.COLOR_RGB2BGR)

with mp_hands.Hands(
    max_num_hands=2,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.5,
) as hands:
    results = hands.process(test_frame_rgb)  # MediaPipe requires RGB

if results.multi_hand_landmarks:
    print(f'Detected {len(results.multi_hand_landmarks)} hand(s)')
    for i, (hand_landmarks, handedness) in enumerate(
        zip(results.multi_hand_landmarks, results.multi_handedness)
    ):
        label = handedness.classification[0].label
        score = handedness.classification[0].score
        print(f'  Hand {i}: {label} (confidence {score:.2f})')
        # Print fingertip landmarks
        for tip_id in [4, 8, 12, 16, 20]:
            lm = hand_landmarks.landmark[tip_id]
            print(f'    Landmark {tip_id}: x={lm.x:.3f} y={lm.y:.3f} z={lm.z:.3f}')
else:
    print('No hands detected in first frame — try a different sample frame')

In [ ]:
# Draw landmarks on the frame
if results.multi_hand_landmarks:
    annotated = test_frame_bgr.copy()
    for hand_landmarks in results.multi_hand_landmarks:
        mp_draw.draw_landmarks(
            annotated,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS,
        )
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title('MediaPipe hand landmarks')
    plt.axis('off')
    plt.show()
else:
    print('No landmarks to draw')

## 3. RT-DETR fretboard detection

**What RT-DETR does:** A real-time detection transformer. Unlike YOLO, it uses an end-to-end transformer decoder so there's no NMS post-processing step. It's fast and accurate, and it's available in HuggingFace Transformers.

**Pre-trained vs fine-tuned:**
- The pre-trained model (COCO) does NOT know what a guitar fretboard is.
- You need to fine-tune on your annotated Roboflow dataset.
- Cell below first tests the pre-trained model (it will probably detect 'remote' or nothing useful).
- After fine-tuning, replace the checkpoint path with your local model.

**Roboflow annotation guide:**
1. Record 10–15 diverse frames (different frets, lighting conditions, angles)
2. Upload to Roboflow and draw bounding boxes around the fretboard region
3. Label as class 'fretboard'
4. Export in COCO JSON format for fine-tuning

In [ ]:
# Test the pre-trained RT-DETR (will not detect fretboard — this just verifies the setup)
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor
from PIL import Image
import torch

processor = RTDetrImageProcessor.from_pretrained('PekingU/rtdetr_r50vd')
model     = RTDetrForObjectDetection.from_pretrained('PekingU/rtdetr_r50vd')
model.eval()

pil_frame = Image.fromarray(test_frame_rgb)
inputs = processor(images=pil_frame, return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

# Decode results
results_det = processor.post_process_object_detection(
    outputs,
    threshold=0.5,
    target_sizes=[pil_frame.size[::-1]],
)[0]

print(f'Detections above 0.5 confidence: {len(results_det["scores"])}')
for score, label, box in zip(
    results_det['scores'], results_det['labels'], results_det['boxes']
):
    label_name = model.config.id2label[label.item()]
    x1, y1, x2, y2 = box.tolist()
    print(f'  {label_name:<20} conf={score:.2f}  box=({x1:.0f},{y1:.0f},{x2:.0f},{y2:.0f})')

In [ ]:
# TODO: after fine-tuning, test your fretboard model here
# Replace checkpoint path with your local model directory

# FRETBOARD_MODEL_PATH = '../models/rtdetr_fretboard'
# processor_ft = RTDetrImageProcessor.from_pretrained(FRETBOARD_MODEL_PATH)
# model_ft     = RTDetrForObjectDetection.from_pretrained(FRETBOARD_MODEL_PATH)
# model_ft.eval()

# inputs = processor_ft(images=pil_frame, return_tensors='pt')
# with torch.no_grad():
#     outputs = model_ft(**inputs)
# results_ft = processor_ft.post_process_object_detection(
#     outputs, threshold=0.7, target_sizes=[pil_frame.size[::-1]]
# )[0]
# print('Fine-tuned fretboard detections:', results_ft)

print('Uncomment above after fine-tuning your RT-DETR model')

## 4. Map fingertips to string/fret coordinates

**The key transform:** Once you have:
- The fretboard bounding box (from RT-DETR)
- The fingertip pixel positions (from MediaPipe, in normalized [0,1] coords)

You can compute which string and fret each fingertip is pressing.

**Simplest approach (implement this first):**
- The fretboard box gives you a rectangle in pixel space.
- Divide the box height into 6 equal bands → one per string.
- Divide the box width into N equal bands → one per fret.
- Convert fingertip normalized coords to pixels, then bin into (string, fret).

**Better approach (for v2):**
- The fretboard is not a perfect rectangle from all angles (perspective distortion).
- Use OpenCV's `getPerspectiveTransform` with the four corners of the fretboard
  to warp the frame into a rectified top-down view.
- Then the string/fret binning is accurate regardless of camera angle.
- Also note: fret spacing is not uniform — frets get closer together toward the body.

In [ ]:
# Stub: manually define a fretboard box for now (replace with RT-DETR output later)
# Format: (x1, y1, x2, y2) in pixels
# Measure these from your sample frame by inspecting the image

FRETBOARD_BOX = {
    'x1': 100,   # ← adjust to your frame
    'y1': 200,
    'x2': 900,
    'y2': 350,
    'score': 1.0,
}

def box_to_grid(box, num_strings=6, num_frets=12):
    """Divide a fretboard box into a (string, fret) grid.
    
    Returns dict with:
      'string_ys': y-coordinate of each string centerline (pixels)
      'fret_xs':   x-coordinate of each fret centerline (pixels)
    """
    x1, y1, x2, y2 = box['x1'], box['y1'], box['x2'], box['y2']
    string_ys = [y1 + (y2 - y1) * (i + 0.5) / num_strings for i in range(num_strings)]
    fret_xs   = [x1 + (x2 - x1) * (i + 0.5) / num_frets   for i in range(num_frets)]
    return {'string_ys': string_ys, 'fret_xs': fret_xs}

grid = box_to_grid(FRETBOARD_BOX)
print('String y-positions:', [f'{y:.0f}px' for y in grid['string_ys']])
print('Fret   x-positions:', [f'{x:.0f}px' for x in grid['fret_xs'][:6]], '...')

# Visualize the grid on the frame
vis = test_frame_rgb.copy()
for y in grid['string_ys']:
    cv2.line(vis, (FRETBOARD_BOX['x1'], int(y)), (FRETBOARD_BOX['x2'], int(y)), (0, 255, 0), 1)
for x in grid['fret_xs']:
    cv2.line(vis, (int(x), FRETBOARD_BOX['y1']), (int(x), FRETBOARD_BOX['y2']), (255, 0, 0), 1)

plt.figure(figsize=(12, 6))
plt.imshow(vis)
plt.title('Manual fretboard grid (green=strings, blue=frets)')
plt.axis('off')
plt.show()

## 5. Full video pass (to implement in Phase 3)

Once fretboard detection and hand tracking are working on single frames, run the full video here to produce the `vision_frames` list needed by `aitabs.fusion.fuse`.

In [ ]:
# TODO: implement this after single-frame steps above are working

# from aitabs.vision.fretboard import detect_fretboard, load_model, fretboard_coordinates
# from aitabs.vision.hands import build_detector, track_hands, fingertip_to_string_fret

# fret_model, fret_processor = load_model('../models/rtdetr_fretboard')
# hands_detector = build_detector(max_num_hands=2)

# vision_frames = []
# cap = cv2.VideoCapture(VIDEO_PATH)
# fps = cap.get(cv2.CAP_PROP_FPS)
# frame_idx = 0

# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break
#     timestamp = frame_idx / fps

#     fretboard_box = detect_fretboard(frame, fret_model, fret_processor)
#     hands = track_hands(frame, hands_detector)

#     vision_frames.append({
#         'frame_index': frame_idx,
#         'timestamp': timestamp,
#         'fretboard': fretboard_box,
#         'hands': hands,
#     })
#     frame_idx += 1

# cap.release()
# print(f'Processed {len(vision_frames)} frames')

print('Uncomment the block above after implementing the vision submodules')